# FiremeX — train a YOLO26n fire and smoke model (Google Colab)

**Why YOLO26n.** Same size class as our YOLOv8n models, but about twice as fast on a CPU
(Ultralytics: 38.9 ms vs 80.4 ms per picture, ONNX), more accurate on COCO (40.9 vs 37.3 mAP),
**no NMS step** (the step that broke our model-v1 scores on the Mac), and a loss (STAL) aimed at
**small, distant objects** — our weak point is thin, far-away smoke.

**Why it starts from `yolo26n.pt`, not from model-v3.** A different architecture cannot reuse
YOLOv8 weights. The knowledge is in the data, so this run uses **all** of D-Fire instead of
10,000 reminders:

| Pictures | Size | Role |
|---|---|---|
| D-Fire, all training photos | ~17,000 | real fire and smoke + ~45 % with neither (negatives) |
| Home Fire Dataset | 3,900 | indoor rooms — our use case |
| Our EZVIZ camera | 239 | our room, our camera colours, lighter negatives |

**How to run**
1. Runtime → Change runtime type → **T4 GPU**.
2. Runtime → **Run all**. Allow Google Drive when asked, and upload
   **`firemex_own_dataset.zip`** (project folder `ml/datasets/`) when the button appears.
3. Training takes several hours. Progress is saved to Google Drive after every epoch:
   **if Colab disconnects, open the notebook again and Run all — it carries on where it stopped.**
4. The last cells compare YOLO26n with **model-v3** on three test sets and download the new
   model (`.pt` and `.onnx`) only if it is at least as good.

Datasets: D-Fire (github.com/gaiasd/DFireDataset) and Home Fire Dataset
(github.com/PengBo0/Home-fire-dataset, CC BY-NC 4.0 — study use only). Credit both.

In [ ]:
# Google Drive keeps the training progress safe if Colab disconnects.
from google.colab import drive
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/firemex_yolo26"
import os
os.makedirs(SAVE, exist_ok=True)

# YOLO26 needs Ultralytics 8.4. Pinned, so a re-run uses exactly the same code.
!pip -q install ultralytics==8.4.176 kagglehub onnx onnxslim
# wandb would hijack logging; dill makes model files the add-on cannot open.
!pip -q uninstall -y wandb dill
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch to a T4 GPU runtime")

In [ ]:
# 1) Home Fire Dataset from its GitHub release. Its classes are 0 = flame, 1 = smoke;
#    FiremeX uses 0 = smoke, 1 = fire, so every label is swapped (once).
import glob, shutil
BASE = "https://github.com/PengBo0/Home-fire-dataset/releases/download/v1.0.0"
os.makedirs("/content/homefire", exist_ok=True)
for split in ("train", "val", "test"):
    if not os.path.exists(f"/content/homefire/{split}/images"):
        !wget -q -O /content/{split}.zip {BASE}/{split}.zip
        !unzip -q -o /content/{split}.zip -d /content/homefire/{split}
        os.remove(f"/content/{split}.zip")
SWAP = {"0": "1", "1": "0"}
if not os.path.exists("/content/homefire/.swapped"):
    for lab in glob.glob("/content/homefire/*/labels/*.txt"):
        lines = [l.split() for l in open(lab) if l.strip()]
        with open(lab, "w") as fh:
            for l in lines:
                fh.write(" ".join([SWAP[l[0]]] + l[1:]) + "\n")
    open("/content/homefire/.swapped", "w").close()
for split in ("train", "val", "test"):
    print("Home Fire", split, len(glob.glob(f"/content/homefire/{split}/images/*")))

In [ ]:
# 2) D-Fire, ALL of it (classes already 0 = smoke, 1 = fire). Used in place - no copying.
#    If Kaggle asks you to sign in: kaggle.com -> Settings -> Create New Token, upload
#    kaggle.json with the folder icon on the left, and run this cell again.
import kagglehub
root = kagglehub.dataset_download("sayedgamal99/smoke-fire-detection-yolo")
def split_dir(*names):
    for n in names:
        hits = glob.glob(f"{root}/**/{n}/images", recursive=True)
        if hits:
            return hits[0]
dfire = {"train": split_dir("train"), "val": split_dir("val", "valid"), "test": split_dir("test")}
for k, v in dfire.items():
    print("D-Fire", k, v, len(glob.glob(f"{v}/*")) if v else "MISSING")
empty = sum(1 for p in glob.glob(f"{os.path.dirname(dfire['train'])}/labels/*.txt") if os.path.getsize(p) == 0)
print("D-Fire training pictures with no fire or smoke (negatives):", empty)

In [ ]:
# 3) Our camera pictures. Upload firemex_own_dataset.zip when the button appears.
#    Its test clips (take3_02 burning paper, lighter_1m_03) are never trained on.
from google.colab import files
if not os.path.exists("/content/own/train/images"):
    up = files.upload()
    zip_name = next(iter(up))
    !unzip -q -o "/content/{zip_name}" -d /content/own
print("own train", len(glob.glob("/content/own/train/images/*")),
      "| own test", len(glob.glob("/content/own/test/images/*")))

In [ ]:
# 4) One training list. Validation mixes indoor (Home Fire) and outdoor (D-Fire), so
#    the "best" epoch is not chosen by one kind of scene only - last time a model picked
#    on indoor validation got worse on D-Fire.
import yaml
names = {0: "smoke", 1: "fire"}
train = ["/content/homefire/train/images", "/content/own/train/images", dfire["train"]]
val = ["/content/homefire/val/images"] + ([dfire["val"]] if dfire["val"] else [])
yaml.safe_dump({"train": train, "val": val, "names": names}, open("/content/mix.yaml", "w"))
tests = {"Home fire test": "/content/homefire/test/images",
         "Our camera test": "/content/own/test/images",
         "D-Fire test": dfire["test"]}
for label, path in tests.items():
    yaml.safe_dump({"train": path, "val": path, "names": names},
                   open(f"/content/test_{label.split()[0].lower()}.yaml", "w"))
print(open("/content/mix.yaml").read())

In [ ]:
# 5) Train YOLO26n from its COCO weights, with Ultralytics' own YOLO26 recipe
#    (optimizer, learning rate, loss, mosaic) - their guide advises not changing
#    those without a reason. Our two reasons:
#      hsv_h=0.005  colour separates orange fire from other things: keep hue almost fixed
#      flipud=0.0   fire does not burn downwards on CCTV
#    Progress is saved to Drive every epoch; a re-run resumes from it.
from ultralytics import YOLO
RUN = f"{SAVE}/runs/yolo26n"
last = f"{RUN}/weights/last.pt"
if os.path.exists(last):
    print("Resuming from", last)
    try:
        YOLO(last).train(resume=True)
    except AssertionError as exc:          # raised when every epoch is already done
        print("Training already finished:", exc)
else:
    YOLO("yolo26n.pt").train(
        data="/content/mix.yaml", epochs=60, patience=15, imgsz=640, batch=32,
        seed=0, deterministic=True, workers=2,
        hsv_h=0.005, flipud=0.0,
        project=f"{SAVE}/runs", name="yolo26n", exist_ok=True)

In [ ]:
# 6) YOLO26n vs model-v3 (what we would replace) on every test set, scored separately
#    from training. Recall = share of real fire/smoke found; it matters most.
import pandas as pd
!wget -q -O /content/model_v3.pt https://github.com/malika1234m/Firemax/releases/download/model-v3/fire_model.pt
def score(weights, label):
    r = YOLO(weights).val(data=f"/content/test_{label.split()[0].lower()}.yaml", split="val",
                         imgsz=640, batch=32, plots=False, verbose=False)
    per = {r.names[int(i)]: float(r.box.r[k]) for k, i in enumerate(r.box.ap_class_index)}
    return {"mAP50": round(float(r.box.map50), 3),
            "fire recall": round(per.get("fire", float("nan")), 3),
            "smoke recall": round(per.get("smoke", float("nan")), 3)}
candidates = {"model-v3 (YOLOv8n)": "/content/model_v3.pt",
              "YOLO26n best.pt": f"{RUN}/weights/best.pt",
              "YOLO26n last.pt": f"{RUN}/weights/last.pt"}
rows = [{"model": m, "test set": t, **score(w, t)} for m, w in candidates.items() for t in tests]
table = pd.DataFrame(rows)
table.to_csv(f"{SAVE}/comparison.csv", index=False)
display(table.pivot(index="model", columns="test set", values="mAP50"))
display(table)

In [ ]:
# 7) Keep YOLO26n only if it beats model-v3 on average AND is not more than 0.03 worse on
#    any test set AND its smoke recall on Home Fire is not lower (smoke is our weak point).
avg = table.groupby("model")["mAP50"].mean()
new = max(["YOLO26n best.pt", "YOLO26n last.pt"], key=lambda m: avg[m])
by = lambda m, col="mAP50": table[table.model == m].set_index("test set")[col]
old, cand = "model-v3 (YOLOv8n)", new
worse = {t: (by(old)[t], by(cand)[t]) for t in tests if by(cand)[t] < by(old)[t] - 0.03}
smoke_ok = by(cand, "smoke recall")["Home fire test"] >= by(old, "smoke recall")["Home fire test"]
print(f"average mAP50: {old} {avg[old]:.3f} | {cand} {avg[cand]:.3f} | smoke recall kept: {smoke_ok}")
PASSED = avg[cand] > avg[old] and not worse and smoke_ok
print("PASSED - downloading" if PASSED else f"NOT better - keep model-v3. Worse on: {worse or 'average/smoke'}")

In [ ]:
# 8) Save the chosen weights, export ONNX (what a CPU runs fastest), download both.
#    Run this even if the gate failed if you want to test it on our videos anyway:
#    the files are then named *_candidate.
tag = "fire_model_v4_yolo26n" + ("" if PASSED else "_candidate")
shutil.copy(candidates[new], f"/content/{tag}.pt")
YOLO(f"/content/{tag}.pt").export(format="onnx", imgsz=640, simplify=True)   # writes /content/{tag}.onnx
for f in (f"/content/{tag}.pt", f"/content/{tag}.onnx"):
    shutil.copy(f, SAVE)
    !sha256sum "{f}"
    files.download(f)